# Use the engine

Use an {class}`~liesel.goose.EngineBuilder` when you need an explicit schedule
or want to add kernels yourself. For routine runs, start with
{doc}`sampling`.

In [1]:
import jax.numpy as jnp
import numpy as np
import tensorflow_probability.substrates.jax.bijectors as tfb
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.goose as gs
import liesel.model as lsl

## Prepare the example

These examples require the regression `model`, including its transformation
and inference specifications, from {doc}`tutorials/md/01c-transform`.

In [2]:
# Python cell fragment: imports are visible in each guide.
# Simulate regression observations.
rng = np.random.default_rng(42)
n = 500
true_beta = np.array([1.0, 2.0])
true_sigma = 1.0
x = rng.uniform(size=n)
X_mat = np.column_stack([np.ones(n), x])
y_vec = X_mat @ true_beta + rng.normal(scale=true_sigma, size=n)

# Define the priors.
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, 0.0, 5.0),
    name="beta",
)
sigma_sq = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.InverseGamma, concentration=3.0, scale=2.0),
    name="sigma_sq",
)

# Connect the parameters to the observations.
sigma = lsl.Var.new_calc(jnp.sqrt, sigma_sq, name="sigma")
X = lsl.Var.new_obs(X_mat, name="X")
mu = lsl.Var.new_calc(jnp.dot, X, beta, name="mu")
y = lsl.Var.new_obs(
    y_vec,
    dist=lsl.Dist(tfd.Normal, mu, sigma),
    name="y",
)

sigma_sq.biject(tfb.Exp(), name="log_sigma_sq")
log_sigma_sq = sigma_sq.bijected_var

joint = gs.MCMCSpec(
    gs.NUTSKernel,
    kernel_group="regression",
    jitter_dist=tfd.Normal(0.0, 0.2),
)
beta.inference = joint
log_sigma_sq.inference = joint
model = lsl.Model(y)


## Build a schedule

For the `model` from {doc}`tutorials/md/01c-transform`:

In [3]:
builder = gs.LieselMCMC(model).get_engine_builder(seed=2026, num_chains=4)
builder.positions_included = ["sigma_sq"]

builder.add_adaptation(1000)
builder.add_burnin(200)
builder.add_posterior(2000)
builder.show_progress = False
engine = builder.build()

liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


In [4]:
engine.sample_all_epochs()
results = engine.get_results()

liesel.goose.engine - INFO - Finished warmup


{meth}`get_engine_builder <liesel.goose.LieselMCMC.get_engine_builder>` supplies the model interface, initial state, kernels,
and configured jitter. The remaining calls choose the schedule, construct
the engine, run it, and retrieve its results.

## Customize adaptation

Use a new builder to change the fast and slow adaptation windows:

In [5]:
builder = gs.LieselMCMC(model).get_engine_builder(seed=2026, num_chains=4)
builder.add_adaptation(
    1000,
    init=100,
    term=100,
    base=50,
)
builder.add_posterior(1000)
builder.show_progress = False
engine = builder.build()

liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


In [6]:
engine.sample_all_epochs()
results = engine.get_results()

liesel.goose.engine - INFO - Finished warmup


Integer `init` and `term` values specify numbers of iterations; floats
specify fractions of the adaptation duration. `base` sets the first slow
window. See {meth}`~liesel.goose.EngineBuilder.add_adaptation` for the schedule.
Add adaptation before burnin and posterior sampling.

## Supply a log density

Goose can also sample a log density represented without a Liesel graph.
This independent example targets a standard normal variable:

In [7]:
def log_prob(state):
    return -0.5 * jnp.square(state["x"]).sum()


builder = gs.EngineBuilder(seed=2026, num_chains=4)
builder.set_model(gs.DictInterface(log_prob))
builder.set_initial_values({"x": jnp.array(0.0)})
builder.add_kernel(gs.NUTSKernel(["x"]))

builder.add_adaptation(1000)
builder.add_posterior(1000)
builder.show_progress = False
engine = builder.build()

liesel.goose.builder - WARNING - No jitter functions provided. The initial values won't be jittered


liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


In [8]:
engine.sample_all_epochs()
normal_results = engine.get_results()

liesel.goose.engine - INFO - Finished warmup


Here {meth}`set_initial_values <liesel.goose.EngineBuilder.set_initial_values>` broadcasts one state to all chains. For different
starts, pass state leaves with a leading chain axis and
`multiple_chains=True`. The interface provides log-density evaluation and
position updates. See `ModelInterface` for the protocol.

The same engine can use other representations through
`ModelInterface`. For custom update logic, start with
an {doc}`MH proposal <tutorials/md/08-custom-kernel>` before implementing a
complete kernel class.